In [1]:
!pip install -U transformers peft accelerate torch librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 59.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
%cd /content/drive/MyDrive/ASR-whisper-finetuning/Audios

/content/drive/MyDrive/ASR-whisper-finetuning/Audios


# Transcribing **audio>30s**

In [2]:
import librosa
import numpy as np

def transcribe_long_audio(file_path, asr_pipeline, chunk_length=30, lan=None):
    audio, sr = librosa.load(file_path, sr=16000)

    total_duration = len(audio) / sr
    print(f"Total duration: {total_duration:.2f} seconds")

    if total_duration <= chunk_length:
        result = asr_pipeline(file_path)
        return result["text"]

    print("Audio longer than 30s → Splitting into chunks...")

    chunk_samples = chunk_length * sr

    num_chunks = int(np.ceil(len(audio) / chunk_samples))

    full_transcription = []
    x=0

    for i in range(num_chunks):
        if x==0:
          start = i * chunk_samples
          x=1
        else:
          start = (i*chunk_samples)-20000
        end = start + chunk_samples
        chunk_audio = audio[start:end]

        if i == num_chunks - 1:
            end = len(audio)

        print(f"Processing chunk {i+1}/{num_chunks} of the time {start}:{end}...")

        use_timestamp = True if (end - start) / sr > chunk_length else False

        if lan != None:
          result = asr_pipeline(
              chunk_audio,
              return_timestamps=use_timestamp,
              language = "urdu"
          )
        else:
          result = asr_pipeline(
              chunk_audio,
              return_timestamps=use_timestamp
          )

        print(result)
        full_transcription.append(result["text"])

    merged_text = " \n".join(full_transcription)

    return merged_text

# **kingabzpro**

## whisper-**large-v3**-urdu

In [ ]:
from transformers import pipeline

transcriber = pipeline("automatic-speech-recognition", model="kingabzpro/whisper-large-v3-urdu")

transcriber.model.generation_config.forced_decoder_ids = None
transcriber.model.generation_config.language = "ur"

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

In [17]:
MP3_output = transcribe_long_audio("CSS_News.mp3", transcriber)
WAV_output = transcribe_long_audio("CSS_News.wav", transcriber)

print(f"MP3_output: \n{MP3_output}")
print(f"WAV_output: \n{WAV_output}")

# print(transcriber("CSS_News_short.mp3"))
# print(transcriber("CSS_News_short.wav"))

Total duration: 269.98 seconds
Audio longer than 30s → Splitting into chunks...
Processing chunk 1/9 of the time 0:480000...
{'text': 'جو میں نے بات کی کہ بڑا خواب ہوتا ہے کسی کا پاکستان میں سرکاری نوکری اور پھر بیروکریسی کے اندر جانا اور وہاں پر جا کے سروف کرنا لیکن اس میں انیشیلی کیونکہ پورے سسٹم کی مختلف چیزوں پر ہم بات کر رہے ہیں تو سب سے پہلے جو سوال اٹھتا ہے وہ یہی اٹھتا ہے کہ جناب یہ سی ایس ایس کے امتحانات میں جو بنیادی طور پر صرف انگریزی زبان میں لیا جاتا ہے اس میں اردو بھی ہونا چاہیے اس کو بیلنگول ہونا چاہیے کیونکہ یہ تجربہ جو ہے انڈیا میں کیا گیا'}
Processing chunk 2/9 of the time 460000:940000...
{'text': 'اور بہت اچھے نتائج مضبط نتائج وہاں کے نکلیں اب پاکستان میں مسئلہ یہ ہے کہ اردو رابطے کی زبان ضرور ہے لیکن نظامت کی زبان جو ہے پاکستان میں انگریزی کو مانا جاتا ہے تو یہ کتنا بڑا گیپ ہے اس پورے اگر کیونکہ رانگ اٹ میری بگننگ ہو جاتی ہے سی ایس ایس میں اسی حوالے سے سر کیا کہیں گے کیا موقف ہے آپ کا'}
Processing chunk 3/9 of the time 940000:1420000...
{'text': 'اپ کے سوال کی طرف 

# Model trained on **Google Fleurs URDU** Language

### 1. **Kingabz --> 30% Degraded Dataset --> Large-v3-urdu**

```
Learning Rate = 5e-5
```

| Epoch | Training Loss | WER  |
| ----- | ------------- | ---- |
| 1     | 0.36          | 0.57 |
| 2     | 0.25          | 0.76 |
| 3     | 0.21          | 0.71 |


In [3]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from transformers import pipeline

# Path to your LoRA folder
lora_path = r"/content/drive/MyDrive/ASR-whisper-finetuning/kingabz_large_v3_30Downgrade"

# Load base Whisper model (VERY IMPORTANT: must match training base model)
# openai/whisper-large-v3
base_model_id = "kingabzpro/whisper-large-v3-urdu"

processor = WhisperProcessor.from_pretrained(base_model_id)
model = WhisperForConditionalGeneration.from_pretrained(base_model_id)

# Attach LoRA adapter
model = PeftModel.from_pretrained(model, lora_path)

model.eval()

# Create ASR pipeline
asr = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0 if torch.cuda.is_available() else -1
)

# Force language if needed
model.generation_config.language = "ur"
model.generation_config.forced_decoder_ids = None

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [4]:
# Transcribe
MP3_output = transcribe_long_audio("CSS_News.mp3", asr)
WAV_output = transcribe_long_audio("CSS_News.wav", asr)

print(f"MP3_output: \n{MP3_output}")
print(f"WAV_output: \n{WAV_output}")

# print(asr("CSS_News_short.mp3"))
# print(asr("CSS_News_short.wav"))

Total duration: 269.98 seconds
Audio longer than 30s → Splitting into chunks...
Processing chunk 1/9 of the time 0:480000...


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


{'text': 'جو میں نے باتی کہ بڑااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااا'}
Processing chunk 2/9 of the time 460000:940000...
{'text': 'بہت اچھے نتائج مضبت نتائج وہاں کے نکلیں اب پاکستان میں مسلہ یہ ہے کہ اردو رابطے کی زبان ضرور ہے لیکن نظامت کی زبان پاکستان میں انگریزی کو مانا جاتا ہے تو یہ کتنا بڑا گیپ ہے اس پورے کیونکہ رونگ اٹ میری بگنگ ہو جاتی ہے سی ایس ایس میں اسی حوالے سے سر کیا کہیں گے کیا موقف ہے آپ کا'}
Processing chunk 3/9 of the time 940000:1420000...
{'text': 'تھیک ہوں اور جو کینڈیٹس ہیں ایسپائرنٹس ہیں ان تک یہ بات پہنچا سکیں تینکی ویری مچ اب آتا ہوں آپ کے سوال کی طرف دو باتیں ہیں ایک تو ہے لینگویج آف دی انٹرویو ایک ہے لینگویج آف دی ریٹن پیپرز دونوں اتفاق سے انگلیش ہے دونوں میں اب یہ ایک کولونیل لیگیسی ہے جو ابھی تک چلی آ رہی ہے لیکن میں یہ ضرور کہوں گا کہ'}
Processing chunk 4/

Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


{'text': 'اب اس کا کرائیٹیریا یا اس کے پاس جو گدر سنگی ہے اگر مجھے اجازت نہیں ہے لفز یوز کرنے کی بالکل ہے بالکل ہے وہ صرف یہ ہے کہ اس کے ماں باپ کے پاس پیسہ تھا کہ انہوں نے اس کو باہر جا بھیج کے پڑھا لیا اور ایک دوسرا قریب باپ کا بچہ ہے جو گورنمنٹ ڈگری کالنگ سے آگے جا نہیں سکتا قابل ہے وہ لیکن اپنے آپ کو ایکسپریس نہیں کر سکتا ایک فورن لینگوچ میں اس بچارے کا کیا قصور ہےتو اس لیے میں اسم اب یہ کہہ کروں گا کہ بائی لنگویل کمس کم انٹریویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویو کی', 'chunks': [{'timestamp': (0.0, 7.0), 'text': 'اب اس کا کرائیٹیریا یا اس کے پاس جو گدر سنگی ہے اگر مجھے اجازت نہیں ہے لفز یوز کرنے کی'}, {'timestamp': (7.0, 8.0), 'text': ' بالکل ہ

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{'text': 'جو میں نے باتی کہ بڑااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااااا'}
Processing chunk 2/9 of the time 460000:940000...
{'text': 'بہت اچھے نتائج مضبط نتائج وہاں کے نکلیں اب پاکستان میں مسلہ یہ ہے کہ اردو رابطے کی زبان ضرور ہے لیکن نظامت کی زبان پاکستان میں انگریزی کو مانا جاتا ہے تو یہ کتنا بڑا گیپ ہے اس پورے کیونکہ رونگ اٹ میری بگنگ ہو جاتی ہے سی ایس ایس میں اسی حوالے سے سر کیا کہیں گے کیا موقع ہے آپ کا'}
Processing chunk 3/9 of the time 940000:1420000...
{'text': 'دیک ہوں اور جو کینیڈس ہیں ایسپائرنٹس ہیں ان تک یہ بات کنچا سکیں تینکی ویری مچ اب آتا ہوں آپ کے سوال کی طرف دو باتیں ہیں ایک تو ہے لینگویج آف دی انٹرویو ایک ہے لینگویج آف دی رٹن پیپرز دونوں اتفاق سے انگلیش ہیں دونوں میں اب یہ ایک کولونیل لیگیسی ہے جو ابھی تک چلی آ رہی ہے لیکن میں یہ ضرور کہوں گا کہ'}
Processing chunk 4/9 o

Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


{'text': 'اب اس کا کرائیٹیریا یا اس کے پاس جو گدر سنگی ہے اگر مجھے اجازت نہیں ہے لبس یوز کرنے کی بالکل ہے بالکل ہے وہ صرف یہ ہے کہ اس کے ماں باپ کے پاس پیسہ تھا کہ انہوں نے اس کو باہر جا بھیج کے پڑھا لیا اور ایک دوسرا قریب باپ کا بچہ ہے جو گورنمنٹ ڈگری کالنگ سے آگے جا نہیں سکتا قابل ہے وہ لیکن اپنے آپ کو ایکسپریس نہیں کر سکتا ایک فورن لینگوچ میں اس بچارے کا کیا قصور ہےتو اس لیے میں سم اپ یہ کہہ کروں گا کہ بائی لنگوی کمس کم انٹریویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویویو کی', 'chunks': [{'timestamp': (0.0, 7.0), 'text': 'اب اس کا کرائیٹیریا یا اس کے پاس جو گدر سنگی ہے اگر مجھے اجازت نہیں ہے لبس یوز کرنے کی'}, {'timestamp': (7.0, 8.0), 'text': ' بالکل ہے 

### 2. **Kingabz --> 30% Degraded Dataset --> Large-v3-urdu**

```
Learning Rate = 1e-5
```

| Epoch | Training Loss | WER    |
| ----- | ------------- | ----   |
| 1     | 0.5428        | 0.2196 |
| 2     | 0.3369        | 0.2105 |
| 3     | 0.3121        | 0.2349 |
| 4     | 0.2863        | 2.6805 |
| 5     | 0.2694        | 1.6252 |


Best Model -- Epoch 2

In [4]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from transformers import pipeline

# Path to your LoRA folder
lora_path = r"/content/drive/MyDrive/ASR-whisper-finetuning/kingabz_large_v3_30Downgrade_LR"

# Load base Whisper model (VERY IMPORTANT: must match training base model)
# openai/whisper-large-v3
base_model_id = "kingabzpro/whisper-large-v3-urdu"

processor = WhisperProcessor.from_pretrained(base_model_id)
model = WhisperForConditionalGeneration.from_pretrained(base_model_id)

# Attach LoRA adapter
model = PeftModel.from_pretrained(model, lora_path)

model.eval()

# Create ASR pipeline
asr_30D = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0 if torch.cuda.is_available() else -1
)

# Force language if needed
model.generation_config.language = "ur"
model.generation_config.forced_decoder_ids = None

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [7]:
# Transcribe
MP3_output = transcribe_long_audio("CSS_News.mp3", asr_30D)
WAV_output = transcribe_long_audio("CSS_News.wav", asr_30D)

print(f"MP3_output: \n{MP3_output}")
print(f"WAV_output: \n{WAV_output}")

# print(asr("CSS_News_short.mp3"))
# print(asr("CSS_News_short.wav"))

Total duration: 269.98 seconds
Audio longer than 30s → Splitting into chunks...
Processing chunk 1/9 of the time 0:480000...


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


{'text': 'جو میں نے بات کی کہ بڑا خواب ہوتا ہے ہر کسی کا پاکستان میں سرکاری نوکری اور پھر بیروکریسی کے اندر جانا اور وہاں پر جا کے سرو کرنا لیکن اس میں انیشیلی کیونکہ پورے سسٹم کی مختلف چیزوں پر ہم بات کر رہے ہیں تو سب سے پہلے جو سوال اٹھتا ہے وہ یہی اٹھتا ہے کہ جناب یہ سی ایس ایس کے امتحانات میں جو بنیادی طور پر صرف انگریزی زبان میں لیا جاتا ہے اس میں اردو بھی ہونا چاہیے اس کو بیلنگول ہونا چاہیے کیونکہ یہ تجربہ جو ہے انڈیا میں کیا گیا اور بہت اچھے'}
Processing chunk 2/9 of the time 460000:940000...
{'text': 'اور بہت اچھے نتائج مضبط نتائج وہاں کے نکلیں اب پاکستان میں مسئلہ یہ ہے کہ اردو رابطے کی زبان ضرور ہے لیکن نظامت کی زبان جو ہے پاکستان میں انگریزی کو مانا جاتا ہے تو یہ کتنا بڑا گیپ ہے اس پورے کیونکہ رانگ اٹ میری بگننگ ہو جاتی ہے سی ایسی حوالے سے سر کیا کہیں گے کیا موقف ہے آپ کا'}
Processing chunk 3/9 of the time 940000:1420000...
{'text': 'اپ کے سوال کی طرف دو باتیں ہیں ایک تو ہے لینگویج آف دی انٹرویو ایک ہے لینگویج آف دی رٹن پیپرز دونوں اتفاق سے انگلیش ہے دونوں میں اب یہ ایک کولو

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{'text': 'جو میں نے بات کی کہ بڑا خواب ہوتا ہے ہر کسی کا پاکستان میں سرکاری نوکری اور پھر بیروکریسی کے اندر جانا اور وہاں پر جا کے سرو کرنا لیکن اس میں انیشیلی کیونکہ پورے سسٹم کی مختلف چیزوں پر ہم بات کر رہے ہیں تو سب سے پہلے جو سوال اٹھتا ہے وہ یہی اٹھتا ہے کہ جناب یہ سی ایس ایس کے امتحانات میں جو بنیادی طور پر صرف انگریزی زبان میں لیا جاتا ہے اس میں اردو بھی ہونا چاہیے اس کو بیلنگول ہونا چاہیے کیونکہ یہ تجربہ جو ہے انڈیا میں کیا گیا'}
Processing chunk 2/9 of the time 460000:940000...
{'text': 'اور بہت اچھے نتائج مضبط نتائج وہاں کے نکلیں اب پاکستان میں مسئلہ یہ ہے کہ اردو رابطے کی زبان ضرور ہے لیکن نظامت کی زبان جو ہے پاکستان میں انگریزی کو مانا جاتا ہے تو یہ کتنا بڑا گیپ ہے اس پورے کیونکہ رانگ اٹ میری بگننگ ہو جاتی ہے سی ایسی حوالے سے سر کیا کہیں گے کیا موقف ہے آپ کا'}
Processing chunk 3/9 of the time 940000:1420000...
{'text': 'اپ کے سوال کی طرف دو باتیں ہیں ایک تو ہے لینگویج آف دی انٹرویو ایک ہے لینگویج آف دی رٹن پیپرز دونوں اتفاق سے انگلیش ہیں دونوں میں اب یہ ایک کولونیل لیگسی ہے

### 3. **Kingabz --> 0% Degraded Dataset --> Large-v3-urdu**

```
Learning Rate = 1e-5
```

| Epoch | Training Loss | WER  |
| ----- | ------------- | ---- |
| 1     | 0.36          | 0.57 |
| 2     | 0.25          | 0.76 |
| 3     | 0.21          | 0.71 |


In [3]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from transformers import pipeline

# Path to your LoRA folder
lora_path = r"/content/drive/MyDrive/ASR-whisper-finetuning/kingabz_large_v3_0Downgrade_LR"

# Load base Whisper model (VERY IMPORTANT: must match training base model)
# openai/whisper-large-v3
base_model_id = "kingabzpro/whisper-large-v3-urdu"

processor = WhisperProcessor.from_pretrained(base_model_id)
model = WhisperForConditionalGeneration.from_pretrained(base_model_id)

# Attach LoRA adapter
model = PeftModel.from_pretrained(model, lora_path)

model.eval()

# Create ASR pipeline
asr_0D = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0 if torch.cuda.is_available() else -1
)

# Force language if needed
model.generation_config.language = "ur"
model.generation_config.forced_decoder_ids = None

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [5]:
# Transcribe
MP3_output = transcribe_long_audio("CSS_News.mp3", asr_0D)
WAV_output = transcribe_long_audio("CSS_News.wav", asr_0D)

print(f"MP3_output: \n{MP3_output}")
print(f"WAV_output: \n{WAV_output}")

# print(asr("CSS_News_short.mp3"))
# print(asr("CSS_News_short.wav"))

Total duration: 269.98 seconds
Audio longer than 30s → Splitting into chunks...
Processing chunk 1/9 of the time 0:480000...
{'text': 'جو میں نے بات کی کہ بڑا خواب ہوتا ہے کسی کا پاکستان میں سرکاری نوکری اور پھر بیروکریسی کے اندر جانا اور وہاں پر جا کے سرو کرنا لیکن اس میں انیشیلی کیونکہ پورے سسٹم کی مختلف چیزوں پر ہم بات کر رہے ہیں تو سب سے پہلے جو سوال اٹھتا ہے وہ یہی اٹھتا ہے کہ جناب یہ سی ایس ایس کے امتحانات میں جو بنیادی طور پر صرف انگریزی زبان میں لیا جاتا ہے اس میں اردو بھی ہونا چاہیے اس کو بیلنگول ہونا چاہیے کیونکہ یہ تجربہ جو ہے انڈیا میں کیا گیا'}
Processing chunk 2/9 of the time 460000:940000...
{'text': 'اور بہت اچھے نتائج مضبط نتائج وہاں کے نکلیں اب پاکستان میں مسئلہ یہ ہے کہ اردو رابطے کی زبان ضرور ہے لیکن نظامت کی زبان پاکستان میں انگریزی کو مانا جاتا ہے تو یہ کتنا بڑا گیپ ہے اس پورے کیونکہ رانگ اٹ میری بگننگ ہو جاتی ہے سی ایسی حوالے سے سر کیا کہیں گے کیا موقف ہے آپ کا'}
Processing chunk 3/9 of the time 940000:1420000...
{'text': 'اپ کے سوال کی طرف دو باتیں ہیں ایک تو ہے